# Supplementary analyses for the ViT pneumonia report: full hash audit and patient-disjoint fine-tuning

This notebook closes two gaps left by the main notebook (`vit_finetuning_v4.ipynb`, RUN_ID 20261027-120212):

1. **Hash audit for every partition.** The main run stored per-image hash distances only for the 624 test images. Here all 5,840 images (train, validation, test) are compared with the Hugging Face source dataset, the near-identical threshold is varied (0 to 16 bits), and near-duplicates *inside* the Kaggle training folder are counted.
2. **Patient-disjoint, duplicate-free fine-tuning.** Experiments B, B2 and C are repeated with three seeds each on a new training/validation split in which (a) byte-identical duplicate files are removed, (b) no filename-derived patient proxy is shared between training and validation, and (c) near-identical images (perceptual-hash distance at most `NEAR_GROUP_BITS`) are kept on the same side of the split. The official 624-image test folder is unchanged.

**What this cannot do.** File numbering restarts between the Kaggle train and test folders, so patients shared between training and test cannot be identified from file names; that limitation stays.

**Inputs.** The main run's results bundle (`vit_results_bundle_20261027-120212.zip`: it supplies the split manifest with file hashes, the pinned checkpoint revision and the image-level seed results). The Kaggle dataset is downloaded again and every file is checked against the manifest's SHA-256 values, so the images are provably the ones used in the main run.

**Runtime.** About 10 minutes for Section 3 (CPU) and about 50 minutes for Section 4 (T4 GPU; nine fine-tuning runs). Set `RUN_GROUPED_SEEDS = False` to run only the audit.

Everything this notebook writes carries its own id (`SUPP_RUN_ID`) and records the main `RUN_ID` it depends on.

In [ ]:
!pip -q install transformers kagglehub opencv-python-headless scikit-learn scipy huggingface_hub

## 1. Setup and inputs

In [ ]:
import os, gc, json, random, time, glob, re, hashlib, zipfile, datetime, platform, sys
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import numpy as np, pandas as pd, cv2, torch, torch.nn as nn
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
import matplotlib.pyplot as plt
from transformers import AutoConfig, AutoImageProcessor, AutoModelForImageClassification, get_cosine_schedule_with_warmup

SUPP_RUN_ID = "supp-" + datetime.datetime.utcnow().strftime("%Y%m%d-%H%M%S")
RESULTS_DIR = "/content/results_supp"; os.makedirs(RESULTS_DIR, exist_ok=True)
print("SUPP_RUN_ID:", SUPP_RUN_ID)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

# ---- locate and unpack the main run's results bundle
MAIN_BUNDLE_ZIP = "/content/vit_results_bundle_20261027-120212.zip"      # upload this file to Colab first (left panel -> Files)
if not os.path.exists(MAIN_BUNDLE_ZIP):
    try:
        from google.colab import files
        up = files.upload(); MAIN_BUNDLE_ZIP = "/content/" + list(up.keys())[0]
    except Exception as e:
        raise FileNotFoundError(f"Upload the main run's results bundle zip to /content first ({e})")
MAIN_DIR = "/content/main_bundle"
with zipfile.ZipFile(MAIN_BUNDLE_ZIP) as z:
    z.extractall(MAIN_DIR)
MAIN_DIR = os.path.dirname(next(os.path.join(dp, f) for dp, _, fs in os.walk(MAIN_DIR) for f in fs if f == "run_manifest.json"))
main_manifest = json.load(open(f"{MAIN_DIR}/run_manifest.json"))
MAIN_RUN_ID, MODEL_REV = main_manifest["run_id"], main_manifest["model_revision"]
MODEL_NAME = main_manifest["model"]
print("Main run:", MAIN_RUN_ID, "| checkpoint:", MODEL_NAME, "| pinned revision:", MODEL_REV)

man = pd.read_csv(f"{MAIN_DIR}/split_manifest.csv")
print(man.partition.value_counts().to_dict(), "images in the main run's manifest")

# ---- hyper-parameters: identical to the main run
IMG_SIZE, BATCH_SIZE, EVAL_BATCH = 224, 16, 32
FT_EPOCHS, FT_PATIENCE, FT_LR, WEIGHT_DECAY = 5, 2, 2e-5, 0.01
CLAHE_CLIP, CLAHE_TILE = 2.0, (8, 8)
MIN_VAL_ACC = 0.60
SEED = 42; set_seed(SEED)
CLASS_NAMES = ["NORMAL", "PNEUMONIA"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); use_cuda = device.type == "cuda"
print("Device:", device)

def cleanup():
    gc.collect()
    if use_cuda: torch.cuda.empty_cache()

## 2. Download the dataset and verify it against the main run
Every file is located through its relative path in the main run's manifest and its SHA-256 is compared with the stored value, so any later change to the Kaggle files would be detected.

In [ ]:
import kagglehub
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
root = next(dp for dp, dn, fn in os.walk(path) if {"train", "test"}.issubset(set(dn)))
print("Dataset root:", root)
man["abs_path"] = man.rel_path.apply(lambda r: os.path.join(root, r))
missing = [p for p in man.abs_path if not os.path.exists(p)]
assert not missing, f"{len(missing)} files from the main run's manifest are missing, e.g. {missing[:3]}"

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()
with ThreadPoolExecutor(8) as ex:
    sha_now = list(ex.map(sha256_file, man.abs_path))
bad = int((pd.Series(sha_now) != man.sha256.values).sum())
print(f"SHA-256 check: {len(man) - bad} of {len(man)} files identical to the main run" + ("" if bad == 0 else f"  <-- {bad} DIFFER: the Kaggle files changed; interpret with care"))
SHA_MISMATCHES = bad

def patient_proxy(path):
    """Filename-derived patient proxy; meaningful only within one Kaggle folder."""
    b = os.path.basename(path)
    m = re.match(r"person(\d+)_(bacteria|virus)_", b, flags=re.I)
    if m: return f"person{m.group(1)}_{m.group(2).lower()}"
    m = re.match(r"NORMAL2-IM-(\d+)", b, flags=re.I)
    if m: return "N2-IM-" + m.group(1)
    m = re.match(r"IM-(\d+)", b, flags=re.I)
    if m: return "IM-" + m.group(1)
    return None
man["patient_proxy"] = man.abs_path.apply(patient_proxy)
print("file names without a recognised proxy:", int(man.patient_proxy.isna().sum()))

## 3. Hash audit of every partition
All 5,840 images and the 5,829 Hugging Face images (`keremberke/chest-xray-classification`) are converted to 256-bit perceptual hashes (DCT of a 64x64 grayscale thumbnail). For each Kaggle image we store the nearest Hamming distance to each Hugging Face split (`hf_match_all.csv`), vary the near-identical threshold, and count near-identical pairs **inside** the Kaggle training folder (validation to train, test to train, train to train). A distance of 0 to 2 bits is a near-certain duplicate; unrelated X-rays sit at roughly 40 to 80 bits.

In [ ]:
POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

def phash256(gray):
    small = cv2.resize(gray, (64, 64), interpolation=cv2.INTER_AREA).astype(np.float32)
    low = cv2.dct(small)[:16, :16].flatten()
    return np.packbits(low > np.median(low[1:]))

def hamming_min(Q, R, chunk=128, exclude_self=False):
    """Nearest Hamming distance from each row of Q (n x 32 uint8) to the rows of R. exclude_self: Q and R are the same array."""
    out = []
    for s in range(0, len(Q), chunk):
        q = Q[s:s + chunk]
        d = POP[np.bitwise_xor(q[:, None, :], R[None, :, :])].sum(-1).astype(np.int32)
        if exclude_self:
            for r in range(len(q)): d[r, s + r] = 10 ** 6
        out.append(d.min(1))
    return np.concatenate(out)

def hash_paths(paths):
    return np.stack([phash256(cv2.imread(p, cv2.IMREAD_GRAYSCALE)) for p in paths])

import zipfile as _zf
HF_DATASET = "keremberke/chest-xray-classification"
def iter_hf_gray():
    """Yield (split, grayscale image) for every image of the Hugging Face dataset (zip, loose-image or parquet layout)."""
    from huggingface_hub import snapshot_download
    local = snapshot_download(repo_id=HF_DATASET, repo_type="dataset")
    files = [os.path.join(dp, f) for dp, _, fs in os.walk(local) for f in fs]
    def split_of(s):
        s = s.lower()
        return "train" if "train" in s else ("valid" if ("valid" in s or "/val" in s) else ("test" if "test" in s else "unknown"))
    for z in [f for f in files if f.lower().endswith(".zip")]:
        out = os.path.join("/content/hf_cxr", os.path.basename(z)[:-4]); os.makedirs(out, exist_ok=True)
        with _zf.ZipFile(z) as zf: zf.extractall(out)
        files += [os.path.join(dp, f) for dp, _, fs in os.walk(out) for f in fs]
    imgs = [f for f in files if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    if imgs:
        for f in imgs:
            g = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
            if g is not None: yield split_of(f), g
        return
    parquet = [f for f in files if f.lower().endswith(".parquet")]
    if not parquet:
        local = snapshot_download(repo_id=HF_DATASET, repo_type="dataset", revision="refs/convert/parquet")
        parquet = [os.path.join(dp, f) for dp, _, fs in os.walk(local) for f in fs if f.lower().endswith(".parquet")]
    for f in parquet:
        df_ = pd.read_parquet(f); col = "image" if "image" in df_.columns else df_.columns[0]
        for v in df_[col]:
            b = v["bytes"] if isinstance(v, dict) else v
            g = cv2.imdecode(np.frombuffer(b, np.uint8), cv2.IMREAD_GRAYSCALE)
            if g is not None: yield split_of(f), g

In [ ]:
THRESHOLDS = [0, 2, 4, 8, 12, 16]
NEAR = 8

# --- hashes of every Kaggle image of the main run (manifest order)
KH = hash_paths(man.abs_path.tolist())
# --- hashes of the Hugging Face dataset
hf_hash, hf_split = [], []
for sp, g in iter_hf_gray():
    hf_hash.append(phash256(g)); hf_split.append(sp)
HF = np.stack(hf_hash); hf_split = np.array(hf_split)
print("Hugging Face images hashed:", len(HF), {str(s): int((hf_split == s).sum()) for s in np.unique(hf_split)})
np.savez_compressed(f"{RESULTS_DIR}/hashes.npz", kaggle=KH, hf=HF, hf_split=hf_split)

# --- nearest distance of every Kaggle image to each Hugging Face split and to all of it
dist = {"d_hf_any": hamming_min(KH, HF)}
for s in ["train", "valid", "test"]:
    dist[f"d_hf_{s}"] = hamming_min(KH, HF[hf_split == s]) if (hf_split == s).any() else np.full(len(KH), 999)
match = man[["partition", "rel_path", "label"]].copy()
for k, v in dist.items(): match[k] = v
match.to_csv(f"{RESULTS_DIR}/hf_match_all.csv", index=False)

# --- threshold sensitivity
rows = []
for part in ["train", "val", "test"]:
    m = match[match.partition == part]
    for t in THRESHOLDS:
        tr_, va_, te_ = m.d_hf_train <= t, m.d_hf_valid <= t, m.d_hf_test <= t
        rows.append(dict(partition=part, threshold_bits=t, images=len(m), copy_anywhere=int((m.d_hf_any <= t).sum()), copy_in_HF_train=int(tr_.sum()),
                         only_in_HF_valid=int((va_ & ~tr_).sum()), only_in_HF_test=int((te_ & ~tr_ & ~va_).sum()),
                         share_anywhere=round(float((m.d_hf_any <= t).mean()), 4), share_in_HF_train=round(float(tr_.mean()), 4)))
sens_df = pd.DataFrame(rows); sens_df.to_csv(f"{RESULTS_DIR}/hash_threshold_sensitivity.csv", index=False)
display(sens_df)

# --- near-identical pairs inside the Kaggle training folder (and test vs train, as a control)
tr_idx = np.where(man.partition == "train")[0]; va_idx = np.where(man.partition == "val")[0]; te_idx = np.where(man.partition == "test")[0]
d_va_tr = hamming_min(KH[va_idx], KH[tr_idx]); d_te_tr = hamming_min(KH[te_idx], KH[tr_idx]); d_tr_tr = hamming_min(KH[tr_idx], KH[tr_idx], exclude_self=True)
within = {}
for name, d in [("validation_to_train", d_va_tr), ("test_to_train (control)", d_te_tr), ("train_to_train", d_tr_tr)]:
    within[name] = dict(images=int(len(d)), min=int(d.min()), p5=float(np.percentile(d, 5)), median=float(np.median(d)),
                        **{f"within_{t}_bits": int((d <= t).sum()) for t in THRESHOLDS})
within_df = pd.DataFrame(within).T; display(within_df)
json.dump(within, open(f"{RESULTS_DIR}/within_kaggle_near_duplicates.json", "w"), indent=2)
print("Validation images with a near-identical (<= %d bits) training image inside the Kaggle train folder: %d of %d" % (NEAR, within["validation_to_train"][f"within_{NEAR}_bits"], len(va_idx)))

## 4. Patient-disjoint, duplicate-free split
1. **Pool** = the 5,216 files of the Kaggle training folder (the main run's train + validation partitions).
2. **Byte-identical duplicates** (same SHA-256) are reduced to one file.
3. **Groups** are formed so that related images always stay on the same side: images share a group if they have the same filename-derived patient proxy *or* a perceptual-hash distance of at most `NEAR_GROUP_BITS` (default 8) to each other (connected components).
4. A **stratified group split** (`StratifiedGroupKFold`, 7 folds, seed 42, first fold as validation, about 14%) gives training and validation sets with no shared group and a similar class mix.
5. The official 624-image **test folder is unchanged**.

In [ ]:
NEAR_GROUP_BITS = 8
pool = man[man.partition.isin(["train", "val"])].sort_values("rel_path").reset_index()          # 'index' = row in man / KH
n0 = len(pool)
dup_sizes = pool.groupby("sha256").size()
pool = pool.drop_duplicates("sha256", keep="first").reset_index(drop=True)
print(f"pool: {n0} files -> {len(pool)} after removing {n0 - len(pool)} byte-identical duplicates ({int((dup_sizes > 1).sum())} duplicated files)")

# connected components over (same patient proxy) or (hash distance <= NEAR_GROUP_BITS)
parent = list(range(len(pool)))
def find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]; a = parent[a]
    return a
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb: parent[rb] = ra
first_of = {}
for i, pxy in enumerate(pool.patient_proxy):
    if pxy is None or (isinstance(pxy, float) and np.isnan(pxy)): continue
    if pxy in first_of: union(i, first_of[pxy])
    else: first_of[pxy] = i
PH = KH[pool["index"].values]
n_near_pairs = 0
for s in range(0, len(PH), 128):
    d = POP[np.bitwise_xor(PH[s:s + 128][:, None, :], PH[None, :, :])].sum(-1)
    for r in range(d.shape[0]): d[r, s + r] = 999
    ii, jj = np.where(d <= NEAR_GROUP_BITS)
    for a, b in zip(ii + s, jj):
        if a < b: union(int(a), int(b)); n_near_pairs += 1
pool["group"] = [find(i) for i in range(len(pool))]
print(f"near-identical pairs (<= {NEAR_GROUP_BITS} bits) merged into groups: {n_near_pairs}; groups: {pool.group.nunique()} for {len(pool)} images; largest group: {pool.group.value_counts().max()} images")

y_pool = (pool.label == "PNEUMONIA").astype(int).values
sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
tr_i, va_i = next(sgkf.split(pool.rel_path, y_pool, pool.group))
pool["gpartition"] = "train"; pool.loc[va_i, "gpartition"] = "val"
gtr, gva = pool[pool.gpartition == "train"], pool[pool.gpartition == "val"]

# ---- checks
assert not (set(gtr.group) & set(gva.group)), "a group straddles train and validation"
assert not (set(gtr.sha256) & set(gva.sha256)), "a byte-identical file straddles train and validation"
shared_px = set(gtr.patient_proxy.dropna()) & set(gva.patient_proxy.dropna())
assert not shared_px, f"{len(shared_px)} patient proxies are shared"
# near-identical pairs across the new partitions (should be none at <= NEAR_GROUP_BITS)
d_cross = hamming_min(KH[gva["index"].values], KH[gtr["index"].values])
print(f"checks passed: no shared group / file / patient proxy. Nearest hash distance from a validation image to the training set: min {int(d_cross.min())} bits")
summary = pd.DataFrame({"images": [len(gtr), len(gva)], "NORMAL": [int((gtr.label == "NORMAL").sum()), int((gva.label == "NORMAL").sum())],
                        "PNEUMONIA": [int((gtr.label == "PNEUMONIA").sum()), int((gva.label == "PNEUMONIA").sum())]}, index=["train", "val"])
summary["pneumonia:normal"] = (summary.PNEUMONIA / summary.NORMAL).round(2); display(summary)
pool[["rel_path", "label", "patient_proxy", "sha256", "group", "gpartition"]].to_csv(f"{RESULTS_DIR}/grouped_split_manifest.csv", index=False)
GROUPED_COUNTS = summary.to_dict("index")

## 5. Model, label mapping and data loaders (identical to the main run)

In [ ]:
def finetune(use_clahe, tag, use_weights=True, seed=None, data=None):
    if seed is not None:
        set_seed(seed)                       # fixed seed for the repeated-seed experiments
    _tr_p, _tr_lm, _va_p, _va_lm = data if data is not None else (tr_p, tr_lm, va_p, va_lm)
    model = AutoModelForImageClassification.from_pretrained(MODEL_NAME, revision=MODEL_REV).to(device)
    train_loader = make_loader(_tr_p, _tr_lm, use_clahe, True, BATCH_SIZE)
    val_loader   = make_loader(_va_p, _va_lm, use_clahe, False, EVAL_BATCH)

    counts = np.bincount(_tr_lm, minlength=2)
    if use_weights:
        w = counts.sum() / (2.0 * counts)                   # inverse-frequency class weights (model label space)
        print(f"[{tag}] class weights (model id order): {np.round(w, 4)}")
        criterion = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=device))
    else:
        print(f"[{tag}] NO class weights (plain cross-entropy)")
        criterion = nn.CrossEntropyLoss()

    opt = torch.optim.AdamW(model.parameters(), lr=FT_LR, weight_decay=WEIGHT_DECAY)
    total_steps = FT_EPOCHS * len(train_loader)
    sched = get_cosine_schedule_with_warmup(opt, int(0.1 * total_steps), total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=use_cuda)

    hist = dict(train_loss=[], val_loss=[], val_acc=[])
    best_val, best_state, bad = float("inf"), None, 0

    for ep in range(1, FT_EPOCHS + 1):
        t0 = time.time(); model.train(); run, n = 0.0, 0
        for x, y in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_cuda):
                loss = criterion(model(pixel_values=x).logits.float(), y)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run += loss.item() * x.size(0); n += x.size(0)

        model.eval(); vl, vc, vn = 0.0, 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_cuda):
                    out = model(pixel_values=x).logits.float()
                vl += criterion(out, y).item() * x.size(0)
                vc += (out.argmax(1) == y).sum().item(); vn += x.size(0)
        tl, vl, va = run / n, vl / vn, vc / vn
        hist["train_loss"].append(tl); hist["val_loss"].append(vl); hist["val_acc"].append(va)
        print(f"[{tag}] epoch {ep}/{FT_EPOCHS} | train_loss {tl:.4f} | val_loss {vl:.4f} | val_acc {va:.4f} | {time.time()-t0:.0f}s")

        if vl < best_val:
            best_val, bad = vl, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= FT_PATIENCE:
                print(f"[{tag}] early stopping at epoch {ep}"); break

    model.load_state_dict(best_state)
    if max(hist["val_acc"]) < MIN_VAL_ACC:
        print(f"[{tag}] WARNING: validation accuracy never reached {MIN_VAL_ACC} - this run failed, re-run it.")
    return model, hist

def plot_hist(h, title):
    plt.figure(figsize=(6, 3.5))
    plt.plot(h["train_loss"], label="train loss"); plt.plot(h["val_loss"], label="val loss")
    plt.title(title); plt.xlabel("epoch"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
cfg = AutoConfig.from_pretrained(MODEL_NAME, revision=MODEL_REV)
processor = AutoImageProcessor.from_pretrained(MODEL_NAME, revision=MODEL_REV)
MEAN, STD = processor.image_mean, processor.image_std
name2id = {str(v).upper(): int(k) for k, v in cfg.id2label.items()}
assert set(name2id) == {"NORMAL", "PNEUMONIA"}, name2id
ds2model = [name2id[c] for c in CLASS_NAMES]; POS = name2id["PNEUMONIA"]
lab2m = lambda s: [ds2model[CLASS_NAMES.index(l)] for l in s]
print("label mapping:", cfg.id2label, "| POS id:", POS)

def apply_clahe(gray_np):
    return cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_TILE).apply(gray_np)

class XrayDS(Dataset):
    def __init__(self, paths, labels, use_clahe=False, train=False):
        self.paths, self.labels, self.use_clahe = paths, labels, use_clahe
        self.aug = T.Compose([T.RandomHorizontalFlip(), T.RandomRotation(10)]) if train else T.Compose([])
        self.to_tensor = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("L").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        arr = np.array(img)
        if self.use_clahe: arr = apply_clahe(arr)
        img = Image.fromarray(arr).convert("RGB"); img = self.aug(img)
        return self.to_tensor(img), self.labels[i]

def make_loader(paths, labels, use_clahe, train, bs):
    return DataLoader(XrayDS(paths, labels, use_clahe, train), batch_size=bs, shuffle=train, num_workers=2, pin_memory=use_cuda)

@torch.no_grad()
def get_probs(model, loader):
    model.eval(); out, ys = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_cuda):
            logits = model(pixel_values=x).logits
        out.append(torch.softmax(logits.float(), dim=1)[:, POS].cpu().numpy()); ys.append((y == POS).numpy().astype(int))
    return np.concatenate(out), np.concatenate(ys)

def quick_metrics(probs, ys):
    pred = (probs >= 0.5).astype(int)
    tp = int(((pred == 1) & (ys == 1)).sum()); tn = int(((pred == 0) & (ys == 0)).sum())
    fp = int(((pred == 1) & (ys == 0)).sum()); fn = int(((pred == 0) & (ys == 1)).sum())
    return dict(accuracy=(tp + tn) / len(ys), sensitivity=tp / max(tp + fn, 1), specificity=tn / max(tn + fp, 1),
                auc=float(roc_auc_score(ys, probs)), FN=fn, FP=fp)

# official test folder (unchanged) in the main run's order
te = man[man.partition == "test"].reset_index(drop=True)
te_p, te_lm = te.abs_path.tolist(), lab2m(te.label)
test_raw   = make_loader(te_p, te_lm, False, False, EVAL_BATCH)
test_clahe = make_loader(te_p, te_lm, True,  False, EVAL_BATCH)
# new patient-disjoint training / validation data
g_tr_p, g_tr_lm = [os.path.join(root, r) for r in gtr.rel_path], lab2m(gtr.label)
g_va_p, g_va_lm = [os.path.join(root, r) for r in gva.rel_path], lab2m(gva.label)
GROUPED_DATA = (g_tr_p, g_tr_lm, g_va_p, g_va_lm)
# the original image-level split, for a like-for-like reference accuracy of the unchanged checkpoint on the new validation set
ref = AutoModelForImageClassification.from_pretrained(MODEL_NAME, revision=MODEL_REV).to(device)
pv, yv = get_probs(ref, make_loader(g_va_p, g_va_lm, False, False, EVAL_BATCH))
pt, yt = get_probs(ref, test_raw)
A_VAL_GROUPED = quick_metrics(pv, yv); A_TEST = quick_metrics(pt, yt)
print("Reference checkpoint A on the patient-disjoint validation set:", {k: round(v, 4) for k, v in A_VAL_GROUPED.items()})
print("Reference checkpoint A on the official test folder:", {k: round(v, 4) for k, v in A_TEST.items()})
del ref; cleanup()

## 6. Fine-tuning B, B2 and C on the patient-disjoint split (three seeds each)
Same recipe as the main run (`finetune` is copied unchanged): AdamW, learning rate 2e-5, five epochs at most, early stopping on validation loss, class weights for B and C only, CLAHE for C only. Seeds are fixed in advance (11, 22, 33), every run is reported, and the test set is not used to choose anything. Results are saved after every run.

In [ ]:
RUN_GROUPED_SEEDS = True
SEEDS = [11, 22, 33]
CONFIGS = {"B": dict(use_clahe=False, use_weights=True), "B2": dict(use_clahe=False, use_weights=False), "C": dict(use_clahe=True, use_weights=True)}
rows, histories, test_probs = [], {}, {}
if RUN_GROUPED_SEEDS:
    for tag, kw in CONFIGS.items():
        for s in SEEDS:
            model_s, hist_s = finetune(use_clahe=kw["use_clahe"], tag=f"{tag}-grouped-seed{s}", use_weights=kw["use_weights"], seed=s, data=GROUPED_DATA)
            probs_s, ys_s = get_probs(model_s, test_clahe if kw["use_clahe"] else test_raw)
            b = int(np.argmin(hist_s["val_loss"]))
            r = dict(split="patient-disjoint", config=tag, seed=s, best_epoch=b + 1, best_val_loss=hist_s["val_loss"][b], val_acc_at_best=hist_s["val_acc"][b], **quick_metrics(probs_s, ys_s))
            rows.append(r); histories[f"{tag}_s{s}"] = hist_s; test_probs[f"p_{tag}_s{s}"] = probs_s; print(r)
            pd.DataFrame(rows).to_csv(f"{RESULTS_DIR}/grouped_seed_runs.csv", index=False)
            json.dump(histories, open(f"{RESULTS_DIR}/grouped_histories.json", "w"), indent=2)
            pd.concat([te[["rel_path"]], pd.DataFrame(test_probs)], axis=1).to_csv(f"{RESULTS_DIR}/grouped_predictions_test.csv", index=False)
            del model_s; cleanup()
else:
    print("Grouped fine-tuning skipped.")

## 7. Comparison with the image-level split of the main run

In [ ]:
if rows:
    g = pd.DataFrame(rows)
    img = pd.read_csv(f"{MAIN_DIR}/seed_repeats.csv"); img["split"] = "image-level (main run)"
    both = pd.concat([img, g], ignore_index=True)
    ms = both.groupby(["config", "split"])[["val_acc_at_best", "accuracy", "sensitivity", "specificity"]].agg(["mean", "std"]).round(4)
    display(ms); ms.to_csv(f"{RESULTS_DIR}/grouped_vs_image_level_summary.csv")
    tab = []
    for c in CONFIGS:
        a, b = img[img.config == c], g[g.config == c]
        tab.append({"Config": c,
                    "Val acc, image-level split %": f"{100*a.val_acc_at_best.mean():.2f} \u00b1 {100*a.val_acc_at_best.std():.2f}",
                    "Val acc, patient-disjoint %": f"{100*b.val_acc_at_best.mean():.2f} \u00b1 {100*b.val_acc_at_best.std():.2f}",
                    "Test acc, image-level split %": f"{100*a.accuracy.mean():.2f} \u00b1 {100*a.accuracy.std():.2f}",
                    "Test acc, patient-disjoint %": f"{100*b.accuracy.mean():.2f} \u00b1 {100*b.accuracy.std():.2f}",
                    "Gap val-test, patient-disjoint (pp)": round(100 * (b.val_acc_at_best.mean() - b.accuracy.mean()), 2)})
    cmp_df = pd.DataFrame(tab); display(cmp_df); cmp_df.to_csv(f"{RESULTS_DIR}/grouped_vs_image_level_table.csv", index=False)
    print(f"Reference checkpoint A: test accuracy {100*A_TEST['accuracy']:.2f}%, accuracy on the patient-disjoint validation set {100*A_VAL_GROUPED['accuracy']:.2f}%")
    fig, ax = plt.subplots(figsize=(6, 3.6))
    for i, c in enumerate(CONFIGS):
        for sp, col, off in [("image-level (main run)", "#c0622f", -0.12), ("patient-disjoint", "#1f4e79", 0.12)]:
            v = both[(both.config == c) & (both.split == sp)].accuracy.values * 100
            ax.plot([i + off] * len(v), v, "o", color=col, label=sp if i == 0 else None)
    ax.axhline(100 * A_TEST["accuracy"], color="k", ls="--", lw=0.8, label="A (reference)")
    ax.set_xticks(range(len(CONFIGS))); ax.set_xticklabels(list(CONFIGS)); ax.set_ylabel("test accuracy (%)"); ax.legend(fontsize=7); plt.tight_layout(); plt.show()

## 8. Export

In [ ]:
import shutil
SUPP_MANIFEST = dict(supp_run_id=SUPP_RUN_ID, depends_on_main_run_id=MAIN_RUN_ID, model=MODEL_NAME, model_revision=MODEL_REV,
                     sha256_mismatches_vs_main_run=SHA_MISMATCHES, near_threshold_bits=NEAR, near_group_bits=NEAR_GROUP_BITS,
                     seeds=SEEDS, hyperparameters=dict(epochs=FT_EPOCHS, patience=FT_PATIENCE, lr=FT_LR, weight_decay=WEIGHT_DECAY, batch=BATCH_SIZE),
                     grouped_split_counts=GROUPED_COUNTS if "GROUPED_COUNTS" in globals() else None,
                     environment=dict(python=sys.version.split()[0], torch=torch.__version__, cuda=torch.version.cuda, opencv=cv2.__version__,
                                      gpu=(torch.cuda.get_device_name(0) if use_cuda else None)))
json.dump(SUPP_MANIFEST, open(f"{RESULTS_DIR}/supp_manifest.json", "w"), indent=2)
with open(f"{RESULTS_DIR}/README_supplementary.txt", "w") as f:
    f.write(f"Supplementary results {SUPP_RUN_ID} (depends on main run {MAIN_RUN_ID})\n"
            "hf_match_all.csv                    nearest hash distance of every Kaggle image to the Hugging Face train / valid / test splits\n"
            "hash_threshold_sensitivity.csv      near-identical counts at 0, 2, 4, 8, 12, 16 bits\n"
            "within_kaggle_near_duplicates.json  near-duplicates inside the Kaggle training folder\n"
            "grouped_split_manifest.csv          patient-disjoint, duplicate-free training / validation assignment\n"
            "grouped_seed_runs.csv, grouped_seed_summary / grouped_vs_image_level_*.csv, grouped_predictions_test.csv, grouped_histories.json   fine-tuning on that split\n"
            "supp_manifest.json                  ids, revision, parameters, environment\n")
shutil.make_archive(f"/content/vit_supplementary_bundle_{SUPP_RUN_ID}", "zip", RESULTS_DIR)
print("SUPP_RUN_ID:", SUPP_RUN_ID, "| files:", sorted(os.listdir(RESULTS_DIR)))
try:
    from google.colab import files
    files.download(f"/content/vit_supplementary_bundle_{SUPP_RUN_ID}.zip")
except Exception as e:
    print("Skip download:", e)